# Data Cleaning & Feature Engineering
**Group 29** — Philippe Krischker, Ekin Keskin, Buse Kucukcoban Demircioglu

CDC BRFSS: Nutrition, Physical Activity, and Obesity (2011-2024)
110,880 rows × 33 columns → 102,816 clean rows → 18,112 feature rows

In [14]:
import pandas as pd
import numpy as np

# Load raw CSV. Data_Value_Unit column has mixed types, force string.
df = pd.read_csv(
    "../data/Nutrition_Physical_Activity_Obesity.csv",
    dtype={"Data_Value_Unit": str},
    low_memory=False
)
print(f"Shape: {df.shape}")
df.head(2)

Shape: (110880, 33)


,YearStart,YearEnd,LocationAbbr,LocationDesc,Datasource,Class,Topic,Question,Data_Value_Unit,Data_Value_Type,...,GeoLocation,ClassID,TopicID,QuestionID,DataValueTypeID,LocationID,StratificationCategory1,Stratification1,StratificationCategoryId1,StratificationID1
0,2011,2011,AL,Alabama,Behavioral Risk Factor Surveillance System,Obesity / Weight Status,Obesity / Weight Status,Percent of adults aged 18 years and older who ...,NaN,Value,...,"(32.840571122, -86.631860762)",OWS,OWS1,Q036,VALUE,1,Income,"$15,000 - $24,999",INC,INC1525
1,2011,2011,AL,Alabama,Behavioral Risk Factor Surveillance System,Obesity / Weight Status,Obesity / Weight Status,Percent of adults aged 18 years and older who ...,NaN,Value,...,"(32.840571122, -86.631860762)",OWS,OWS1,Q036,VALUE,1,Income,"$25,000 - $34,999",INC,INC2535


## 1. Data Profiling
Assess column usefulness and missingness before deciding what to drop.

In [15]:
# Missing value rate per column (sorted worst-first)
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({"missing": missing, "pct": missing_pct}).query("missing > 0")

,missing,pct
Total,106920,96.4
Data_Value_Unit,106260,95.8
Sex,102960,92.9
Data_Value_Footnote,97666,88.1
Data_Value_Footnote_Symbol,97666,88.1
Education,95040,85.7
Age(years),87120,78.6
Income,83160,75.0
Race/Ethnicity,79200,71.4
Low_Confidence_Limit,13214,11.9


**Decision**: Drop `Data_Value_Unit` (95.8% missing), `Total` (96.4%), `Data_Value_Alt` (redundant), `Data_Value_Type` (always "Value"), and `Datasource` (constant). These carry no information for our analysis.

In [16]:
DROP_COLS = ["Data_Value_Unit", "Total", "Data_Value_Alt", "Data_Value_Type", "Datasource"]
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
print(f"After column pruning: {df.shape}")  # 110880 x 28

After column pruning: (110880, 28)


## 2. Location Filtering
The dataset includes National aggregates and US territories (Guam, Puerto Rico, Virgin Islands). Our choropleth map only supports the 50 states + DC, so these are excluded.

In [17]:
EXCLUDE = ["National", "Guam", "Puerto Rico", "Virgin Islands"]
before = len(df)
df = df[~df["LocationDesc"].isin(EXCLUDE)].reset_index(drop=True)
print(f"Removed {before - len(df)} rows ({len(df)} remaining)")
print(f"States left: {df['LocationDesc'].nunique()}")  # 51

Removed 8064 rows (102816 remaining)
States left: 51


## 3. Data_Value: European Decimal Format
CDC exports use comma as decimal separator (e.g. "34,8" for 34.8%). The `-` footnote symbol marks suppressed values where sample size was inadequate for reliable estimates.

In [18]:
# Check format
print("Sample values:", df["Data_Value"].dropna().head(8).tolist())

# Suppressed values use "-" footnote
is_suppressed = df["Data_Value_Footnote_Symbol"] == "-"
print(f"Suppressed rows: {is_suppressed.sum()} ({is_suppressed.sum()/len(df)*100:.1f}%)")

# Convert: replace comma with dot, then parse as float
df["Data_Value_Num"] = pd.to_numeric(df["Data_Value"].str.replace(",", "."), errors="coerce")
# Suppressed values become NaN automatically (non-numeric content)
df["is_suppressed"] = is_suppressed
print(f"Numeric values: {df['Data_Value_Num'].notna().sum()}")
print(f"Range: {df['Data_Value_Num'].min():.1f}% - {df['Data_Value_Num'].max():.1f}%")

Sample values: ['34,8', '35,8', '32,3', '34,1', '28,8', '16,3', '27,8', '35,2']
Suppressed rows: 10226 (9.9%)
Numeric values: 92590
Range: 0.9% - 85.3%


**Strategy**: For static views (choropleth, box plots) suppressed values stay NaN. For time-series views we interpolate within each (state × question × stratification) group using linear interpolation with a limit of 3 consecutive missing points. This preserves continuity without fabricating long-range trends.

In [19]:
# Interpolation for time-series: groupby state+question+strat, interpolate over YearStart
df_ts = df.copy().sort_values(["LocationDesc", "Question", "StratificationCategory1", "Stratification1", "YearStart"])
group_cols = ["LocationDesc", "Question", "StratificationCategory1", "Stratification1"]
df_ts["Data_Value_Num"] = df_ts.groupby(group_cols)["Data_Value_Num"].transform(
    lambda x: x.interpolate(method="linear", limit_direction="both", limit=3)
)
still_missing = df_ts["Data_Value_Num"].isna().sum()
recovered = df["Data_Value_Num"].isna().sum() - still_missing
print(f"Recovered via interpolation: {recovered} values")
print(f"Still NaN (insufficient neighbors): {still_missing}")

Recovered via interpolation: 3478 values
Still NaN (insufficient neighbors): 6748


## 4. Stratification Labels
Normalize to ensure consistent filtering across the three topic classes (Obesity, Physical Activity, Fruits/Vegetables). Some labels have trailing spaces.

In [20]:
df["StratificationCategory1"] = df["StratificationCategory1"].str.strip()
df["Stratification1"] = df["Stratification1"].str.strip()

# Verify: count rows per stratification dimension
print(df["StratificationCategory1"].value_counts().to_string())

StratificationCategory1
Race/Ethnicity    29376
Income            25704
Age (years)       22032
Education         14688
Sex                7344
Total              3672


## 5. Save cleaned datasets
Two versions: static (NaN where suppressed) and interpolated (for time-series).

In [21]:
df.to_csv("../data/brfss_cleaned.csv", index=False)
df_ts.to_csv("../data/brfss_interpolated.csv", index=False)
print(f"Cleaned (static):  {len(df):,} rows")
print(f"Cleaned (interpolated): {len(df_ts):,} rows")

Cleaned (static):  102,816 rows
Cleaned (interpolated): 102,816 rows
